# Spark Session Initialization

Initialize the Spark Session used for all DataFrame operations in this notebook.

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import os
import sys

# os.environ["PYSPARK_PYTHON"] = sys.executable
# os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

builder = ( SparkSession.builder \
    .appName("BGG Data Validation") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.LocalLogStore")
            
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

# Paths Configuration

Define all input and output paths used in this notebook.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]
DATA_PATH = PROJECT_ROOT / "data"

SOURCE_PATH = DATA_PATH / "source"
REFERENCE_PATH = DATA_PATH / "reference"

GEO_CSV = SOURCE_PATH / "geography.csv"
VENDORS_CSV = SOURCE_PATH / "vendors.csv"
DELIVERY_CSV = SOURCE_PATH / "delivery.csv"
GOOGLE_ANALYTICS_CSV = SOURCE_PATH / "ga.csv"

## Add project root to sys.path to enable importing functions from utils
sys.path.append(str(Path().resolve().parents[0]))

In [ ]:
GEO_CSV.exists(), VENDORS_CSV.exists(), DELIVERY_CSV.exists(), GOOGLE_ANALYTICS_CSV.exists()

# DataFrame Schemas Definition

Define structured schemas for:
- vendors
- delivery
- google_analytics
- geography

In [ ]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, LongType, StringType, DoubleType, TimestampType, DateType
)


geography_schema = StructType([
    StructField("country_id", IntegerType(), False),
    StructField("country_name", StringType(), False),
    StructField("country_code", StringType(), False),
    StructField("region_id", IntegerType(), False),
    StructField("region_name", StringType(), False),
    StructField("continent_id", IntegerType(), False),
    StructField("continent_name", StringType(), False),
])


google_analytics_schema = StructType([
    StructField("ga_device_type", StringType(), False),
    StructField("ga_source_id", StringType(), False),
])


delivery_schema = StructType([
    StructField("delivery_company_name", StringType(), False),
    StructField("delivery_type", StringType(), False),
])

			
vendors_schema = StructType([
    StructField("vendor_name", StringType(), False),
    StructField("vendor_country", StringType(), False),
    StructField("vendor_city", StringType(), False),
    StructField("vat_number", StringType(), True),
])

# Calendar Reference Table Generation

Generate the '_Calendar_' reference DataFrame with derived date attributes.

In [ ]:
from pyspark.sql.functions import (
    col, explode, sequence, to_date, year, quarter, 
    month, dayofmonth, dayofweek, weekofyear, last_day, 
    lit, sha2, concat_ws, date_format, cast
)

start_date = "1990-01-01"
end_date = "2030-12-31"

calendar_df = (
    spark.createDataFrame([(start_date, end_date)], ["start", "end"])
    .withColumn("date", explode(sequence(to_date(col("start")), to_date(col("end")))))
    .select("date")
    .withColumn("year", year(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("month_name", date_format("date", "MMMM"))
    .withColumn("day", dayofmonth(col("date")))
    .withColumn("day_of_week", dayofweek(col("date")))
    .withColumn("week_of_year", weekofyear(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("is_weekend", col("day_of_week").isin([1,7]))
    .withColumn("is_month_start", dayofmonth(col("date")) == 1)
    .withColumn("is_month_end",last_day(col("date")) == col("date"))
)

# Reading Reference Tables

Load reference CSV files into Spark DataFrames using predefined schemas.

In [ ]:
geo_df = (
    spark.read
    .option("header", True)
    .schema(geography_schema)
    .csv(str(GEO_CSV))
)

In [ ]:
ga_df = (
    spark.read
    .option("header", True)
    .schema(google_analytics_schema)
    .csv(str(GOOGLE_ANALYTICS_CSV))
    .withColumn("ga_id",sha2(concat_ws("||", col("ga_device_type"), col("ga_source_id")), 256))
)

In [ ]:
delivery_df = (
    spark.read
    .option("header", True)
    .schema(delivery_schema)
    .csv(str(DELIVERY_CSV))
    .withColumn("delivery_id",sha2(concat_ws("||", col("delivery_type"), col("delivery_company_name")), 256))
)

In [ ]:
vendors_df = (
    spark.read
    .option("header", True)
    .schema(vendors_schema)
    .csv(str(VENDORS_CSV))
    .withColumn("vendor_id",sha2(concat_ws("||", col("vendor_name"), col("vendor_country"), col("vendor_city")), 256))
)


# Persisting Reference Tables

Writing to `delta` reference tables:
- calendar
- delivery
- vendors
- google_analytics
- geography

In [ ]:
from utils.data_io import save_to_reference



save_to_reference(geo_df, "geography")
save_to_reference(ga_df, "google_analytics")
save_to_reference(vendors_df, "vendors")
save_to_reference(delivery_df, "delivery")
save_to_reference(calendar_df, "calendar")